<a href="https://colab.research.google.com/github/dharshini-dev-hub/bigdata-lab/blob/rdd-operations/Spark_colab_connection_and_RDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [4,964 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,762 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,157 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports 

In [2]:
!wget -q https://archive.apache.org/dist/spark/spark-3.4.1/spark-3.4.1-bin-hadoop3.tgz

In [3]:
!tar xf spark-3.4.1-bin-hadoop3.tgz

In [4]:
import os
os.environ["JAVA_HOME"] = f"/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = f"/content/spark-3.4.1-bin-hadoop3"

In [5]:
!pip install -q pyspark==3.5.1

In [6]:
!pip install findspark

In [7]:
import findspark
findspark.init()

In [8]:
import pyspark

## RDD OPERATIONS


In [9]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
.appName("RDD_operation") \
.getOrCreate()

In [10]:
rdd = spark.sparkContext.parallelize([1, 2, 3, 4])
rdd.collect()

[1, 2, 3, 4]

In [11]:
customer_data = [
"customer_id,name,city,state,country,registration_date,is_active",
"0,customer_0,bangalore,karnataka,India,2023-11-11,True",
"1,customer_1,hyderabad,delhi,India,2023-08-26,True",
"2,customer_2,ahmedabdad,west bengal,India,2023-06-23,True",
"3,customer_3,bangalore,tamil nadu,India,2023-03-24,False",
"4,customer_4,bangalore,gujarat,India,2023-06-06,False",
"5,customer_5,delhi,maharashtra,India,2023-04-19,False",
]

In [12]:
rdd = spark.sparkContext.parallelize(customer_data)
rdd.collect()

['customer_id,name,city,state,country,registration_date,is_active',
 '0,customer_0,bangalore,karnataka,India,2023-11-11,True',
 '1,customer_1,hyderabad,delhi,India,2023-08-26,True',
 '2,customer_2,ahmedabdad,west bengal,India,2023-06-23,True',
 '3,customer_3,bangalore,tamil nadu,India,2023-03-24,False',
 '4,customer_4,bangalore,gujarat,India,2023-06-06,False',
 '5,customer_5,delhi,maharashtra,India,2023-04-19,False']

In [13]:
rdd = spark.sparkContext.textFile('/content/sample_data/first_100_customers.csv')
rdd.take(5)

['customer_id,name,city,state,country,registration_date,is_active',
 '0,Customer_0,Pune,Maharashtra,India,2023-06-29,False',
 '1,Customer_1,Bangalore,Tamil Nadu,India,2023-12-07,True',
 '2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,True',
 '3,Customer_3,Bangalore,Karnataka,India,2023-10-17,False']

In [14]:
rdd.getNumPartitions()

2

In [15]:
header = rdd.first()
header

'customer_id,name,city,state,country,registration_date,is_active'

In [16]:
rdd_without_header = rdd.filter(lambda x: x!=header)
rdd_without_header.take(5)

['0,Customer_0,Pune,Maharashtra,India,2023-06-29,False',
 '1,Customer_1,Bangalore,Tamil Nadu,India,2023-12-07,True',
 '2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,True',
 '3,Customer_3,Bangalore,Karnataka,India,2023-10-17,False',
 '4,Customer_4,Ahmedabad,Karnataka,India,2023-03-14,False']

In [17]:
data = (['sathish',45],['jayasree',30],['deepika',25],['dharshini',20])
columns = ['name','age']

df = spark.createDataFrame(data,columns)
df.show()

+---------+---+
|     name|age|
+---------+---+
|  sathish| 45|
| jayasree| 30|
|  deepika| 25|
|dharshini| 20|
+---------+---+



In [18]:
def parse_row(row):
  field = row.split(',')
  return (
      int(field[0]),
         field[1],
         field[2],
         field[3],
         field[4],
         field[5]
  )

In [19]:
parsed_rdd = rdd_without_header.map(parse_row)
parsed_rdd.take(5)

[(0, 'Customer_0', 'Pune', 'Maharashtra', 'India', '2023-06-29'),
 (1, 'Customer_1', 'Bangalore', 'Tamil Nadu', 'India', '2023-12-07'),
 (2, 'Customer_2', 'Hyderabad', 'Gujarat', 'India', '2023-10-27'),
 (3, 'Customer_3', 'Bangalore', 'Karnataka', 'India', '2023-10-17'),
 (4, 'Customer_4', 'Ahmedabad', 'Karnataka', 'India', '2023-03-14')]

In [20]:
numbers = spark.sparkContext.parallelize([1,2,2,4,5])
rdd_numbers = numbers.map(lambda x:x*2)
rdd_numbers.collect()

[2, 4, 4, 8, 10]

In [21]:
rdd_even = rdd_numbers.filter(lambda x:x%2==0)
rdd_even.collect()

[2, 4, 4, 8, 10]

In [22]:
sentence = spark.sparkContext.parallelize(["My name is dharshini"])
rdd_sentence = sentence.flatMap(lambda x:x.split(' '))
rdd_sentence.collect()

['My', 'name', 'is', 'dharshini']

In [23]:
numbers = spark.sparkContext.parallelize([1,2,2,4,5,5])
rdd_numbers = numbers.distinct()
rdd_numbers.collect()

[2, 4, 1, 5]

In [24]:
a = spark.sparkContext.parallelize([1,2])
b = spark.sparkContext.parallelize([3,4])
c = a.union(b)
c.collect()

[1, 2, 3, 4]

In [25]:
a = spark.sparkContext.parallelize([1,2,2,3,4,5])
b = spark.sparkContext.parallelize([2,4,5])
c = a.intersection(b)
c.collect()

[4, 5, 2]

In [26]:
a.count()

6

In [27]:
rdd = spark.sparkContext.parallelize([1,2,3,4,5])
rdd.reduce(lambda x,y:x+y)

15

In [28]:
rdd_file = spark.sparkContext.parallelize(['my','name','is','dharshini'])
rdd_file.saveAsTextFile("/content/sample_data/output_file_1")

In [29]:
rdd_file.collect()

['my', 'name', 'is', 'dharshini']

In [30]:
numbers = spark.sparkContext.parallelize([1,2,3,4,5])
numbers.cache()

ParallelCollectionRDD[46] at readRDDFromFile at PythonRDD.scala:287

In [31]:
from pyspark import StorageLevel
numb = spark.sparkContext.parallelize([1,2,3,4,5])
numb.persist(StorageLevel.DISK_ONLY)

ParallelCollectionRDD[47] at readRDDFromFile at PythonRDD.scala:287

In [32]:
numb.getNumPartitions()

2

In [33]:
numb.repartition(1)

MapPartitionsRDD[52] at coalesce at NativeMethodAccessorImpl.java:0

In [34]:
rdd_1 = spark.sparkContext.parallelize([1,2,3,4,5])
rdd_2 = rdd_1.map(lambda x:x*2)
rdd_2.toDebugString()

b'(2) PythonRDD[54] at RDD at PythonRDD.scala:53 []\n |  ParallelCollectionRDD[53] at readRDDFromFile at PythonRDD.scala:287 []'

In [35]:
rdd = spark.sparkContext.parallelize([('a',1),('b',2),('a',3)])
rdd.groupByKey().collect()

[('b', <pyspark.resultiterable.ResultIterable at 0x7d5b90e52cd0>),
 ('a', <pyspark.resultiterable.ResultIterable at 0x7d5b90e68e90>)]

In [36]:
rdd = spark.sparkContext.parallelize([('a',1),('b',2),('a',3)])
rdd_num = rdd.reduceByKey(lambda x,y:x+y)
rdd_num.collect()

[('b', 2), ('a', 4)]

In [37]:
word = spark.sparkContext.parallelize(["My name is dharshini She is dharshini"])
word_count = word.flatMap(lambda x:x.split(' ')).map(lambda x:(x,1)).reduceByKey(lambda x,y:x+y)
word_count.collect()

[('name', 1), ('dharshini', 2), ('She', 1), ('My', 1), ('is', 2)]

In [38]:
number = spark.sparkContext.parallelize([1,2,3])
number.countByValue()

defaultdict(int, {1: 1, 2: 1, 3: 1})

In [39]:
fruits_1 = spark.sparkContext.parallelize([('apple',2),('banana',3),('mango',2)])
fruits_2 = spark.sparkContext.parallelize([('apple',2),('banana',3),('mango',3)])
joint_fruits = fruits_1.join(fruits_2)
joint_fruits.collect()

[('apple', (2, 2)), ('banana', (3, 3)), ('mango', (2, 3))]

In [40]:
rdd1 = spark.sparkContext.parallelize([("shirt", 10), ("shoes", 20), ("shirt", 15)])
rdd2 = spark.sparkContext.parallelize([("shirt", "blue"), ("shoes", "black"), ("shoes", "white")])

result = rdd1.cogroup(rdd2)
for key, (vals1, vals2) in result.collect():
    print(key, list(vals1), list(vals2))

shirt [10, 15] ['blue']
shoes [20] ['black', 'white']
